# 08 · 몸 상태로 스트레스 단계 분류하기 — k-최근접 이웃 vs 의사결정트리
교과서 **부록 213~218쪽** 선택 활동입니다. 데이터를 못 구한 모둠의 프로젝트 출발점으로도 씁니다.

맨 먼저 **파일 → 드라이브에 사본 저장**을 하세요(내 드라이브에 저장돼야 고친 내용이 남습니다).
`____` 가 보이면 거기가 문제입니다. 빈칸 **바로 위 💡 힌트**를 읽고 채운 뒤 셀을 실행하세요(Shift + Enter).
교과서는 `files.upload()` 로 파일을 올리지만, 이 노트북은 **선생님 저장소에서 바로 읽어** 올릴 파일이 없습니다.

| 열 | 뜻 |
|---|---|
| Humidity | 피부 습도 |
| Temperature | 체온(화씨) |
| Step count | 걸음 수 |
| Stress Level | **정답** — 0 낮음 · 1 보통 · 2 높음 |

## ① 불러오기와 탐색

In [ ]:
주소 = 'https://raw.githubusercontent.com/richee-pc/AI_cs/main/data/'
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(주소 + 'stress_lysis.csv')
print(df.shape)
print(df.groupby('Stress Level')['Temperature'].agg(['count', 'min', 'max']))
sns.boxplot(data=df, x='Stress Level', y='Humidity')
plt.show()

**생각해 보기** 상자그림을 보면 스트레스 단계마다 습도가 어떻게 다른가요? 이 데이터는 모델이 맞히기 쉬울까요, 어려울까요?

## ② 입력(X)과 정답(y), 훈련·테스트 나누기
💡 정답 열을 뺀 나머지가 X → `drop('Stress Level', axis=1)`. 테스트는 30% → `test_size=0.3`.

In [ ]:
from sklearn.model_selection import train_test_split

print(df.isna().sum().sum(), '개 결측치')
x = df.drop('Stress Level', axis=1)
y = df['Stress Level']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=____, random_state=42, stratify=y)
x_train.shape, x_test.shape

## ③ k-최근접 이웃
💡 분류 모델 이름은 `KNeighborsClassifier`.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = ____(n_neighbors=5)
knn.fit(x_train, y_train)
print('kNN 정확도 :', accuracy_score(y_test, knn.predict(x_test)))

## ④ 의사결정트리 — 그림으로 판단 과정 보기
💡 트리 깊이를 3으로 제한하면 그림이 읽기 쉬워집니다 → `max_depth=3`.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

tree = DecisionTreeClassifier(max_depth=____, random_state=28)
tree.fit(x_train, y_train)
print('트리 정확도 :', accuracy_score(y_test, tree.predict(x_test)))

plt.figure(figsize=(14, 6))
plot_tree(tree, feature_names=list(x.columns), class_names=['low', 'mid', 'high'], filled=True, fontsize=8)
plt.show()

## ⑤ 비교와 해석
- 두 모델의 정확도는? kNN ______ · 트리 ______
- 트리의 맨 위 질문(가장 먼저 나누는 속성)은? ______ → 이 속성이 스트레스를 가장 잘 가른다는 뜻입니다.
- 점수 차가 작다면 어느 모델을 고르겠나요? 이유는?  (힌트: Ⅱ단원 «모델 고르는 다섯 걸음»)

## ⑥ (선택) 모델 저장하고 새 값 넣어 보기

In [ ]:
import joblib
joblib.dump(tree, 'stress_tree.pkl')
model = joblib.load('stress_tree.pkl')

습도, 체온, 걸음 = 20.0, 90.0, 150      # 값을 바꿔 보세요
pred = model.predict(pd.DataFrame([[습도, 체온, 걸음]], columns=x.columns))[0]
print(['스트레스 낮은 단계', '스트레스 조심 단계', '스트레스 주의 단계'][pred])

---
### 정답 예시 (막혔을 때만)
- ② `test_size=0.3` · ③ `KNeighborsClassifier` · ④ `max_depth=3`
- 이 데이터는 단계가 아주 깔끔하게 나뉘어 두 모델 모두 0.99 안팎이 나옵니다. **너무 높으면 의심도 해 보세요** — 실제 측정값일까, 만든 데이터일까? (출처 확인!)